# NS-SHAFT Colab pipeline validation

This notebook is intentionally simulator-only. It never uploads/runs the game executable and never creates a Windows input backend. The optional 768-step PPO cell validates plumbing only. Historical spike BC0 and the current P4.1 S0/S1/S2/S3 experiment are separate, default-disabled bounded cells; DAgger, DQfD and long training remain out of scope.

## Private repository / manual upload setup

The setup cell defaults to manual upload and does not require GitHub credentials. Download or create a ZIP of the repository on your computer, upload that ZIP when prompted, and keep the repository structure intact. The ZIP may contain a top-level folder; the cell locates `pyproject.toml` automatically. If the project was already extracted under `/content`, it is reused without prompting.

In [ ]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
print('Headless SDL configured')

In [ ]:
import os
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

# Default for a private repository: upload one repository ZIP manually.
# If you already extracted the repository in Colab, set this to that folder.
MANUAL_PROJECT_PATH = ''  # Example: '/content/stairkid-rl-main'
CONTENT_ROOT = Path('/content')

def project_candidates(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted({
        marker.parent.resolve()
        for marker in root.rglob('pyproject.toml')
        if (marker.parent / 'src' / 'stair_agent').is_dir()
    })

def archive_contains_project(archive):
    try:
        with zipfile.ZipFile(archive) as bundle:
            names = {name.replace('\\\\', '/') for name in bundle.namelist()}
        markers = [name for name in names if name.endswith('pyproject.toml')]
        return any(
            f"{marker[:-len('pyproject.toml')]}src/stair_agent/" in names
            or any(name.startswith(f"{marker[:-len('pyproject.toml')]}src/stair_agent/") for name in names)
            for marker in markers
        )
    except zipfile.BadZipFile:
        return False

def safe_extract(archive):
    destination = Path(tempfile.mkdtemp(prefix='stairkid-upload-', dir=CONTENT_ROOT))
    destination_root = destination.resolve()
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            target = (destination / item.filename).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f'Unsafe path in ZIP: {item.filename}')
        bundle.extractall(destination)
    return destination

if MANUAL_PROJECT_PATH:
    candidates = project_candidates(Path(MANUAL_PROJECT_PATH).expanduser())
else:
    candidates = project_candidates(CONTENT_ROOT)

if not candidates:
    archives = [path for path in CONTENT_ROOT.glob('*.zip') if archive_contains_project(path)]
    if not archives:
        from google.colab import files
        print('Upload the repository ZIP (not a checkpoint ZIP).')
        uploaded = files.upload()
        archives = [CONTENT_ROOT / name for name in uploaded if archive_contains_project(CONTENT_ROOT / name)]
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly one repository ZIP, found {len(archives)}: {archives}')
    candidates = project_candidates(safe_extract(archives[0]))

if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one NS-SHAFT project, found {len(candidates)}: {candidates}')

REPO_DIR = candidates[0]
os.chdir(REPO_DIR)
print(f'Project located at: {REPO_DIR}')
assert (REPO_DIR / 'pyproject.toml').is_file()
python_version = sys.version_info[:2]
print('Python runtime:', sys.version)
if not ((3, 11) <= python_version < (3, 13)):
    raise RuntimeError(f'Unsupported Python {python_version}; expected Python 3.11 or 3.12')

from importlib.metadata import version
setuptools_version = version('setuptools')
setuptools_major = int(setuptools_version.split('.', 1)[0])
print('Setuptools:', setuptools_version)
if setuptools_major < 65:
    raise RuntimeError('setuptools>=65 is required before installing this project')

# Do not use pip -q here: Colab must show the actual resolver/build error if installation fails.
# Colab already provides setuptools; reusing it avoids a redundant isolated build download.
# Use a normal wheel install so the package is importable in this running kernel immediately.
install_command = [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--no-cache-dir', '--no-build-isolation', '.[test,rl]',
    'tensorboard', 'imageio[ffmpeg]'
]
print('Installing project and dependencies from:', REPO_DIR)
install_result = subprocess.run(install_command, cwd=REPO_DIR)
if install_result.returncode != 0:
    raise RuntimeError(
        f'pip install failed with exit code {install_result.returncode}. '
        'Read the complete pip output immediately above this message.'
    )

import gymnasium
import pymunk
import stable_baselines3
import stair_agent

print('Installed ai-stair-agent:', version('ai-stair-agent'))
print('Gymnasium:', gymnasium.__version__)
print('Pymunk:', version('pymunk'))
print('Stable-Baselines3:', stable_baselines3.__version__)
print('Installation and import checks passed; Windows-only input packages were skipped.')

## Optional Google Drive mount

Run this only when persistent artifacts are needed. Use experiment IDs; do not overwrite a sole `latest` checkpoint.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/ns-shaft-runs') if USE_DRIVE else Path('/content/ns-shaft-runs')
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT

## Tests, Gymnasium contract and headless smoke

In [ ]:
!python -m pytest -q
!python scripts/check_simulator.py --steps 10000 --baseline-steps 1000 --seed 0

In [ ]:
from gymnasium.utils.env_checker import check_env
from stair_agent.envs.shaft_env import ShaftEnv
env = ShaftEnv()
check_env(env, skip_render_check=True)
env.close()
print('check_env passed')

## Sync/async vector benchmark (1, 4, 8, 16 envs)

The recommendation is throughput-based and must still be checked against Colab RAM/CPU utilization.

In [ ]:
import time
import numpy as np
from gymnasium.vector import AsyncVectorEnv, SyncVectorEnv
from stair_agent.envs.shaft_env import ShaftEnv

def make_env():
    return ShaftEnv()

def benchmark(vector_cls, count, vector_steps=2000):
    vector = vector_cls([make_env for _ in range(count)])
    try:
        vector.reset(seed=list(range(count)))
        started = time.perf_counter()
        for _ in range(vector_steps):
            vector.step(np.zeros(count, dtype=np.int64))
        elapsed = time.perf_counter() - started
        return count * vector_steps / elapsed
    finally:
        vector.close()

results = []
for count in (1, 4, 8, 16):
    for name, vector_cls in (('sync', SyncVectorEnv), ('async', AsyncVectorEnv)):
        rate = benchmark(vector_cls, count)
        results.append({'mode': name, 'envs': count, 'steps_per_second': rate})
        print(name, count, f'{rate:.0f} steps/s')
recommended = max(results, key=lambda item: item['steps_per_second'])
print('Throughput recommendation:', recommended)

## Versioned artifact paths, TensorBoard and bounded pipeline validation

In [ ]:
from datetime import datetime, timezone
import json
EXPERIMENT_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_sim_v0_probe')
RUN_DIR = ARTIFACT_ROOT / EXPERIMENT_ID
for name in ('tensorboard', 'checkpoints', 'videos'):
    (RUN_DIR / name).mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'config.json').write_text(json.dumps({
    'experiment_id': EXPERIMENT_ID,
    'observation_schema': 'stair-observation-v3-268',
    'purpose': 'bounded Colab runtime/checkpoint/resume/video validation'
}, indent=2) + '\n')
print(RUN_DIR)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $RUN_DIR/tensorboard

In [ ]:
RUN_COLAB_PIPELINE_VALIDATION = False  # Set True only after tests/benchmark cells pass.
if not RUN_COLAB_PIPELINE_VALIDATION:
    print('Validation disabled: set RUN_COLAB_PIPELINE_VALIDATION=True to run the frozen 768-step smoke.')
else:
    import imageio.v3 as iio
    import numpy as np
    import torch
    from stable_baselines3 import PPO
    from stair_agent.envs.shaft_env import ShaftEnv, ShaftEnvConfig

    train_env = ShaftEnv(config=ShaftEnvConfig(max_episode_steps=120))
    model = PPO(
        'MlpPolicy', train_env, n_steps=128, batch_size=64, n_epochs=1,
        policy_kwargs={'net_arch': [64, 64]}, seed=51001,
        device='auto', tensorboard_log=str(RUN_DIR / 'tensorboard'), verbose=0,
    )
    model.learn(total_timesteps=512, progress_bar=False)
    initial_checkpoint = RUN_DIR / 'checkpoints' / 'ppo_000512'
    model.save(initial_checkpoint)
    loaded = PPO.load(str(initial_checkpoint) + '.zip', env=train_env, device='auto')
    before_resume = loaded.num_timesteps
    loaded.learn(total_timesteps=256, reset_num_timesteps=False, progress_bar=False)
    resumed_checkpoint = RUN_DIR / 'checkpoints' / 'ppo_000768_resumed'
    loaded.save(resumed_checkpoint)
    assert loaded.num_timesteps >= before_resume + 256
    train_env.close()

    video_env = ShaftEnv(
        config=ShaftEnvConfig(max_episode_steps=120), render_mode='rgb_array'
    )
    observation, _ = video_env.reset(seed=52001)
    frames = [video_env.render()]
    for _ in range(120):
        action, _ = loaded.predict(observation, deterministic=True)
        observation, _, terminated, truncated, _ = video_env.step(int(action))
        frames.append(video_env.render())
        if terminated or truncated:
            break
    video_env.close()
    video_path = RUN_DIR / 'videos' / 'ppo_resumed_eval.mp4'
    iio.imwrite(video_path, np.asarray(frames), fps=8, codec='libx264')

    gate = {
        'initial_checkpoint': (str(initial_checkpoint) + '.zip'),
        'resumed_checkpoint': (str(resumed_checkpoint) + '.zip'),
        'video': str(video_path),
        'video_frames': len(frames),
        'torch_device': str(loaded.device),
        'cuda_available': torch.cuda.is_available(),
        'pipeline_pass': all([
            Path(str(initial_checkpoint) + '.zip').is_file(),
            Path(str(resumed_checkpoint) + '.zip').is_file(),
            video_path.is_file(), len(frames) >= 2,
        ]),
    }
    (RUN_DIR / 'colab_pipeline_gate.json').write_text(
        json.dumps(gate, indent=2) + '\n', encoding='utf-8'
    )
    print(json.dumps(gate, indent=2))
    assert gate['pipeline_pass']

## Spike curriculum v0 — bounded BC0 (seeds 0/1/2)

This section rebuilds the ignored dataset from source, then runs the frozen three-seed BC0 protocol. Each run selects among epochs 3/5/8/11/14/17 on seeds 1060–1079 and evaluates once on untouched seeds 1200–1219. It does not clone GitHub, launch the real game, run DAgger, or start long RL training.

In [ ]:
RUN_SPIKE_BC0 = False  # Set True only after the pytest/check_env cells pass.
if not RUN_SPIKE_BC0:
    print('Spike BC0 disabled. Set RUN_SPIKE_BC0=True for the bounded 3-seed experiment.')
else:
    import json
    import shutil
    import subprocess
    import sys
    from datetime import datetime, timezone

    PROJECT_ROOT = REPO_DIR
    spike_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_spike_bc0')
    SPIKE_RUN_DIR = ARTIFACT_ROOT / spike_run_id
    SPIKE_RUN_DIR.mkdir(parents=True, exist_ok=False)

    def run_checked(*args, allowed_codes=(0,)):
        command = [sys.executable, *map(str, args)]
        print('RUN:', ' '.join(command))
        result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)
        if result.returncode not in allowed_codes:
            raise subprocess.CalledProcessError(result.returncode, command)
        return result.returncode

    # JSONL files are intentionally git-ignored, so always rebuild them.
    run_checked('scripts/run_spike_curriculum_gate.py')
    run_checked('scripts/generate_spike_teacher_dataset.py')
    dataset = PROJECT_ROOT / 'artifacts' / 'spike_teacher_dataset_v0.jsonl'
    assert dataset.is_file() and dataset.stat().st_size > 0

    summaries = []
    for seed in (0, 1, 2):
        prefix = f'spike_bc0_colab_seed_{seed}'
        return_code = run_checked(
            'scripts/run_bc0_smoke.py', '--dataset', dataset,
            '--loss', 'hard', '--seed', seed, '--max-epochs', 17,
            '--output-prefix', prefix, '--curriculum', 'spike-v0',
            '--candidate-epochs', '3,5,8,11,14,17',
            '--selection-seed-start', 1060,
            '--final-eval-seed-start', 1200,
            allowed_codes=(0, 3),
        )
        summary_path = PROJECT_ROOT / 'artifacts' / f'{prefix}_smoke_summary.json'
        model_path = PROJECT_ROOT / 'artifacts' / f'{prefix}_model.pt'
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        summaries.append(summary)
        shutil.copy2(summary_path, SPIKE_RUN_DIR / summary_path.name)
        shutil.copy2(model_path, SPIKE_RUN_DIR / model_path.name)
        for candidate_path in (PROJECT_ROOT / 'artifacts').glob(f'{prefix}_epoch_*_model.pt'):
            shutil.copy2(candidate_path, SPIKE_RUN_DIR / candidate_path.name)
        if return_code == 3:
            print(f'Seed {seed} failed its frozen gate; stopping before later seeds.')
            break

    seed_results = []
    for summary in summaries:
        bc = summary['evaluations']['bc0']
        baseline = summary['evaluations']['baseline']
        seed_results.append({
            'seed': summary['seed'],
            'selected_epoch': summary['selected_epoch'],
            'passed': summary['gate']['passed'],
            'bc_mean_floors': bc['mean_floors'],
            'baseline_mean_floors': baseline['mean_floors'],
            'retention': bc['mean_floors'] / max(baseline['mean_floors'], 1e-9),
            'max_action_share': bc['max_action_share'],
            'health_deaths': bc['terminal_reasons'].get('health_depleted', 0),
            'overall_accuracy': summary['test_classification']['accuracy'],
            'spike_visible_records': summary['spike_visible_classification']['records'],
            'spike_visible_accuracy': summary['spike_visible_classification']['accuracy'],
            'spike_target_records': summary['spike_target_classification']['records'],
            'spike_target_accuracy': summary['spike_target_classification']['accuracy'],
        })

    gate = {
        'experiment': 'spike_bc0_colab_v0',
        'dataset': 'spike_teacher_dataset_v0',
        'seeds': seed_results,
        'completed_seed_count': len(seed_results),
        'passed': len(seed_results) == 3 and all(
            item['passed'] and item['health_deaths'] == 0
            and item['retention'] >= 0.80
            and item['max_action_share'] < 0.98
            for item in seed_results
        ),
        'dagger_started': False,
    }
    gate_path = SPIKE_RUN_DIR / 'spike_bc0_colab_gate.json'
    gate_path.write_text(json.dumps(gate, indent=2) + '\n', encoding='utf-8')
    archive = shutil.make_archive(str(SPIKE_RUN_DIR), 'zip', root_dir=SPIKE_RUN_DIR)
    print(json.dumps(gate, indent=2))
    print('Download this archive:', archive)
    if not gate['passed']:
        print('Gate failed or stopped early. Download the archive for diagnosis; do not start DAgger.')

## P4.1 — bounded S0/S1/S2/S3 causal/sequence ablation

Run this only after the setup, pytest, check_env and benchmark cells pass. Use the dedicated local P4.1 bundle because it includes the frozen, git-ignored 3,529-row Spike Teacher Dataset v1; regenerating it with the newer Teacher code is intentionally forbidden. The runner verifies its SHA-256 manifest, trains three initializations with a fixed 300-update budget, selects checkpoints only on seeds 4000–4019, and evaluates the frozen S0/chosen architecture once on seeds 4100–4139. A scientific FAIL is saved normally instead of raising `CalledProcessError`. This cell never controls the Windows game and never starts DAgger, PPO or DQN.

In [ ]:
RUN_P41_ABLATION = False  # Set True only after setup/tests/check_env pass.
if not RUN_P41_ABLATION:
    print('P4.1 disabled. Set RUN_P41_ABLATION=True for the frozen bounded experiment.')
else:
    import json
    import shutil
    import subprocess
    import sys
    from datetime import datetime, timezone

    PROJECT_ROOT = REPO_DIR
    p41_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_p41_ablation')
    P41_RUN_DIR = ARTIFACT_ROOT / p41_run_id
    P41_RUN_DIR.mkdir(parents=True, exist_ok=False)

    def run_p41_command(*args):
        command = [sys.executable, *map(str, args)]
        print('RUN:', ' '.join(command))
        return subprocess.run(command, cwd=PROJECT_ROOT, check=True)

    # This exact frozen JSONL is included only in the dedicated local bundle.
    dataset = PROJECT_ROOT / 'artifacts' / 'spike_teacher_dataset_v1.jsonl'
    if not dataset.is_file():
        raise FileNotFoundError(
            'Missing frozen artifacts/spike_teacher_dataset_v1.jsonl. '
            'Upload the dedicated P4.1 bundle, not a GitHub source ZIP.'
        )
    assert dataset.is_file() and dataset.stat().st_size > 0

    result_dir = P41_RUN_DIR / 'results'
    run_p41_command(
        'scripts/run_p41_ablation.py',
        '--dataset', dataset,
        '--manifest', 'artifacts/p41_experiment_manifest.json',
        '--execute-colab', '--device', 'auto',
        '--output-dir', result_dir,
    )
    summary_path = result_dir / 'p41_ablation_summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    shutil.copy2(PROJECT_ROOT / 'artifacts' / 'p41_experiment_manifest.json', P41_RUN_DIR)
    archive = shutil.make_archive(str(P41_RUN_DIR), 'zip', root_dir=P41_RUN_DIR)
    print(json.dumps({
        'status': summary['status'],
        'selected_architecture': summary['selected_architecture'],
        'next_stage': summary['next_stage'],
    }, indent=2))
    print('Download this archive:', archive)
    if summary['status'] != 'PASS':
        print('Scientific Gate stopped. Download the ZIP for diagnosis; do not start P4.2.')